# models

> Overlay data shapes for the transcript-correction workflow: the Correction /
> CorrectionSession graph nodes + their relation registry, the read view of a
> committed spine segment, the worklist item, run configuration, and the
> correction run manifest (proto-bundle that chains decomp -> correction).

In [ ]:
#| default_exp models

In [ ]:
#| export
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

from pydantic import Field
from cjm_graph_domains.core import DomainNode

In [ ]:
#| export
class CorrectionRelations:
    """Registry of edge types the correction overlay adds to the spine graph."""
    CORRECTS = "CORRECTS"          # Correction -> the layer-0 Segment(s) it corrects
    SUPERSEDES = "SUPERSEDES"      # Correction -> the prior Correction it replaces (undo/update chain)
    DERIVED_FROM = "DERIVED_FROM"  # grouping Correction -> the layer-0 Segments it regroups/prunes
    REVIEWED = "REVIEWED"          # CorrectionSession -> Segment (carries a `decision` property)

    @classmethod
    def all(cls) -> list:  # All relation type strings
        """Return all defined relation types."""
        return [v for k, v in cls.__dict__.items()
                if not k.startswith('_') and isinstance(v, str)]

In [ ]:
#| export
class Correction(DomainNode):
    """A single non-destructive correction over the committed spine (overlay node).

    Layer-0 spine nodes are immutable; every correction is a supersede-able
    overlay. Defined in-core as a `DomainNode` subclass (revolution-1) rather
    than in the shared graph-domain library — its shape is still being validated
    by real runs; the node/edge construction + effective-view projection are
    CR-18 spec material, not yet substrate-owned.
    """
    correction_type: str                                   # "text_content" | "punctuation" | "grouping"
    status: str = "applied"                                # "proposed" | "applied" | "superseded"
    session_id: str = ""                                   # Owning CorrectionSession id
    payload: Dict[str, Any] = Field(default_factory=dict)  # Type-specific data (new text, prune set, ...)
    actor: str = "human"                                   # "human" | "agent:<id>" | "capability:<name>"
    canonical_form: Optional[str] = None                   # Optional entity key (cross-transcript matching)
    rationale: Optional[str] = None                        # Optional human/agent note
    created_at: float = Field(default_factory=time.time)   # Unix timestamp

In [ ]:
#| export
class CorrectionSession(DomainNode):
    """A resumable, reopen-able correction review over one or more documents."""
    status: str = "in_progress"                             # "in_progress" | "completed" | "reopened"
    scope: List[str] = Field(default_factory=list)         # Document node ids in scope
    started_at: float = Field(default_factory=time.time)   # Unix timestamp at session start
    updated_at: float = Field(default_factory=time.time)   # Unix timestamp of last activity

In [ ]:
#| export
@dataclass
class SpineSegment:
    """A committed layer-0 Segment loaded from the graph (read view)."""
    id: str                                   # Graph Segment node id
    index: int                                # 0-based position in the document spine
    text: str                                 # Segment text (may be empty for silence VAD chunks)
    start_time: Optional[float] = None        # Source-coordinate start (seconds)
    end_time: Optional[float] = None          # Source-coordinate end (seconds)
    source_plugin_name: Optional[str] = None  # First SourceRef plugin_name (provenance anchor)
    source_row_id: Optional[str] = None       # First SourceRef row_id (upstream job_id)
    content_hash: Optional[str] = None        # First SourceRef content_hash

    @property
    def is_empty(self) -> bool:  # True when the segment has no non-whitespace text
        """Empty-text segment (silence VAD chunk with no aligned words; decomp D14)."""
        return not (self.text or "").strip()

In [ ]:
#| export
@dataclass
class WorklistItem:
    """One spine segment surfaced for review, with its deterministic Tier-1 flags."""
    segment: SpineSegment                            # The segment under review
    flags: List[str] = field(default_factory=list)   # Tier-1 signal flags (empty, boundary, divergence, ...)

    @property
    def index(self) -> int:  # Segment spine index
        """Spine index of the underlying segment."""
        return self.segment.index

In [ ]:
#| export
@dataclass
class CorrectionConfig:
    """Configuration for one correction run."""
    graph_plugin: str = "cjm-graph-plugin-sqlite"  # Graph-storage capability id
    graph_db_path: Optional[str] = None            # Graph DB the spine lives in (from the decomp manifest)
    actor: str = "human"                           # Actor recorded on corrections + review markers
    assume_yes: bool = False                       # Auto-accept HITL seams (headless mode)
    prune_empty: bool = True                       # Run the D14 empty-segment prune as the first operation

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class CorrectionManifest:
    """Durable record of one correction run (proto-bundle; chains decomp -> correction; CR-20)."""
    run_id: str                  # Unique run identifier
    created_at: float            # Unix timestamp at run start
    config: Dict[str, Any]       # CorrectionConfig snapshot
    decomp_manifest: str         # Path to the consumed decomp run manifest
    graph_db_path: str           # Graph DB the corrections were written to (shared with decomp)
    session_id: str              # CorrectionSession node id
    source_format: str = ""      # Upstream (decomp) manifest format tag
    source_version: str = ""     # Upstream (decomp) manifest schema version
    secondary_manifest: Optional[str] = None  # Optional second-transcriber decomp manifest (diff)
    signals_used: List[str] = field(default_factory=list)          # Signal names contributing to the worklist
    documents: List[Dict[str, Any]] = field(default_factory=list)  # Per-document outcome records

    FORMAT: str = field(default="cjm-transcript-correction-core/run-manifest", repr=False)  # Format tag
    VERSION: str = field(default="0.1.0", repr=False)                                        # Schema version

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
        """Serialize to a plain dict."""
        return {
            "format": self.FORMAT,
            "version": self.VERSION,
            "run_id": self.run_id,
            "created_at": self.created_at,
            "config": self.config,
            "decomp_manifest": self.decomp_manifest,
            "secondary_manifest": self.secondary_manifest,
            "graph_db_path": self.graph_db_path,
            "session_id": self.session_id,
            "source_format": self.source_format,
            "source_version": self.source_version,
            "signals_used": self.signals_used,
            "documents": self.documents,
        }

    def save(
        self,
        path: Union[str, Path],  # Destination JSON file (parent dirs created)
    ) -> Path:  # The written path
        """Write the manifest as pretty-printed JSON."""
        out = Path(path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(self.to_dict(), indent=2))
        return out

In [ ]:
#| export
def new_run_id() -> str:  # e.g. "correct_20260608_153000_1a2b3c4d"
    """Generate a unique, sortable correction run id."""
    return f"correct_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

In [ ]:
# models smoke checks
c = Correction(correction_type="grouping", status="applied", session_id="s1",
               payload={"operation": "prune_empty", "pruned_segment_ids": ["a", "b"]})
node = c.to_graph_node()
assert node.label == "Correction"
assert node.properties["correction_type"] == "grouping"
assert node.properties["payload"]["operation"] == "prune_empty"
assert "id" not in node.properties              # id maps to the structural field, not properties
assert "rationale" not in node.properties       # None excluded by exclude_none

sess = CorrectionSession(scope=["doc1"])
assert sess.to_graph_node().label == "CorrectionSession"
assert sess.to_graph_node().properties["status"] == "in_progress"

assert SpineSegment(id="n1", index=3, text="  ").is_empty
assert not SpineSegment(id="n2", index=4, text="hi").is_empty
assert WorklistItem(segment=SpineSegment(id="n2", index=4, text="hi"), flags=["x"]).index == 4

assert set(CorrectionRelations.all()) == {"CORRECTS", "SUPERSEDES", "DERIVED_FROM", "REVIEWED"}

m = CorrectionManifest(run_id="r", created_at=0.0, config={}, decomp_manifest="/tmp/d.json",
                       graph_db_path="/tmp/g.db", session_id="s1")
assert m.to_dict()["format"] == "cjm-transcript-correction-core/run-manifest"
assert new_run_id().startswith("correct_")
print("models checks OK")

models checks OK
